# Golden Model: 32-bit Polynomial Hardware Core

Generates **exactly 30 test vectors** per run, sampling across 5 test categories:

| Category | What it tests |
|---|---|
| Basic cubic | Small well-behaved coefficients, sweep of small ±x |
| Linear / isolated terms | A=B=0, verifies individual term isolation |
| Saturation | x large enough to overflow INT32 — tests sat vs ovf divergence |
| Mixed signs | Negative leading coefficient, alternating-sign coefficients |
| Zero / constant / boundary x | All-zero poly, constant-only, INT32_MAX/MIN as x |

**Change `BATCH` (cell below) to get a different reproducible set of 30.**

## Polynomial
$$y = Ax^3 + Bx^2 + Cx + D$$

- Inputs $A, B, C, D, x$: 32-bit signed integers  
- Intermediate arithmetic: 64-bit signed (no truncation until final step)  
- `expected_sat`: clamp to `[INT32_MIN, INT32_MAX]`  
- `expected_ovf`: two's complement 32-bit wrap (lower 32 bits as signed int32)

In [1]:
# ── CHANGE THIS to get a different batch of 30 vectors ──
BATCH = 0
# ────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import random
from IPython.display import display

## Golden Model

In [2]:
INT32_MAX =  2**31 - 1
INT32_MIN = -2**31

def poly_golden(A, B, C, D, x):
    for name, val in [('A',A),('B',B),('C',C),('D',D),('x',x)]:
        if not (INT32_MIN <= val <= INT32_MAX):
            raise ValueError(f"{name}={val} out of int32 range")
    result = int(A)*int(x)**3 + int(B)*int(x)**2 + int(C)*int(x) + int(D)
    expected_sat = max(INT32_MIN, min(INT32_MAX, result))
    expected_ovf = int(np.int32(np.uint32(result & 0xFFFFFFFF)))
    return int(expected_sat), int(expected_ovf)

def make_vec(A, B, C, D, x):
    sat, ovf = poly_golden(A, B, C, D, x)
    return dict(A=A, B=B, C=C, D=D, x=x, expected_sat=sat, expected_ovf=ovf)

## Test Case Pool
Five categories, each with a large pool of candidates. 6 are sampled from each category per run.

In [3]:
# ── Category 1: Basic cubic ──────────────────────────────────────────────────
# Small well-behaved coefficients, sweep ±x. Mirrors A=2,B=3,C=4,D=5 family.
pool_basic = []
for A, B, C, D in [(2,3,4,5),(1,2,3,4),(3,1,2,1),(2,2,2,2),(5,4,3,2),
                   (1,0,0,0),(0,1,0,0),(0,0,1,0),(3,0,0,1),(1,1,1,1)]:
    for x in [-3,-2,-1,0,1,2,3]:
        pool_basic.append((A,B,C,D,x))

# ── Category 2: Linear / isolated terms ─────────────────────────────────────
# A=B=0 isolates the C*x+D path. Mirrors 0,0,1,0 and 1,1,1,0 families.
pool_linear = []
for C, D in [(1,0),(2,5),(-3,7),(10,-4),(1,1),(-1,0),(0,100),
             (5,-5),(100,0),(-7,3)]:
    for x in [-10,-5,-1,0,1,5,10]:
        pool_linear.append((0,0,C,D,x))

# ── Category 3: Saturation ───────────────────────────────────────────────────
# 64-bit result exceeds INT32 range. expected_sat != expected_ovf for all.
pool_sat = []
for A, x in [(1,1500),(1,2000),(1,1291),(1,1300),(2,1200),   # positive overflow
             (1,1400),(3,1100),(1,1350),(2,1100),(1,1600)]:
    pool_sat.append((A,0,0,0,x))
for A, x in [(1,-1500),(1,-2000),(1,-1291),(-1,1500),(-1,2000),  # negative overflow
             (1,-1300),(-2,1200),(1,-1400),(-1,1300),(-1,1400)]:
    pool_sat.append((A,0,0,0,x))

# ── Category 4: Mixed signs ──────────────────────────────────────────────────
# Negative leading coeff, alternating signs. Mirrors -1,2,-3,10 family.
pool_mixed = []
for A, B, C, D in [(-1,2,-3,10),(-2,3,-1,5),(-1,-1,1,1),
                   (1,-2,3,-4),(-3,2,-1,0),(-5,10,-2,7),
                   (-1,1,-1,1),(2,-3,2,-1),(-4,3,-2,1),(1,-1,1,-1)]:
    for x in [-4,-2,-1,0,1,2,3]:
        pool_mixed.append((A,B,C,D,x))

# ── Category 5: Zero / constant / boundary x ─────────────────────────────────
# All-zero poly, constant-only (A=B=C=0), INT32_MAX/MIN as x.
pool_zero = []
for x in [-5,-3,-1,0,1,3,5,7]:              # all-zero poly
    pool_zero.append((0,0,0,0,x))
for D in [100,-100,1,-1,999,-50]:            # constant-only
    for x in [-99,0,5]:
        pool_zero.append((0,0,0,D,x))
pool_zero += [                               # boundary x with linear passthrough
    (0,0,1,0,INT32_MAX),(0,0,1,0,INT32_MIN),
    (0,0,0,100,INT32_MAX),(0,0,0,100,INT32_MIN),
    (0,0,0,0,INT32_MAX),(0,0,0,0,INT32_MIN),
]

print(f"Pool sizes — basic:{len(pool_basic)}  linear:{len(pool_linear)}  "
      f"sat:{len(pool_sat)}  mixed:{len(pool_mixed)}  zero:{len(pool_zero)}")

Pool sizes — basic:70  linear:70  sat:20  mixed:70  zero:32


## Sample 30 Vectors (6 per category)

In [4]:
PER_CATEGORY = 6  # 5 × 6 = 30 total

def sample_category(pool, n, seed):
    """Deterministic n-sample from pool, seeded by (BATCH, seed)."""
    rng = random.Random(BATCH * 1000 + seed)
    shuffled = list(pool)
    rng.shuffle(shuffled)
    seen, out = set(), []
    for item in shuffled:
        if item not in seen:
            seen.add(item)
            out.append(item)
        if len(out) == n:
            break
    return out

selected = (
    sample_category(pool_basic,  PER_CATEGORY, seed=1) +
    sample_category(pool_linear, PER_CATEGORY, seed=2) +
    sample_category(pool_sat,    PER_CATEGORY, seed=3) +
    sample_category(pool_mixed,  PER_CATEGORY, seed=4) +
    sample_category(pool_zero,   PER_CATEGORY, seed=5)
)

rows = [make_vec(*p) for p in selected]
df = pd.DataFrame(rows, columns=["A","B","C","D","x","expected_sat","expected_ovf"])

assert len(df) == 30, f"Expected 30 vectors, got {len(df)}"
print(f"BATCH={BATCH}  |  {len(df)} vectors")
print(f"  sat != ovf (overflow):  {(df.expected_sat != df.expected_ovf).sum()}")
print(f"  clamped to INT32_MAX:   {(df.expected_sat == INT32_MAX).sum()}")
print(f"  clamped to INT32_MIN:   {(df.expected_sat == INT32_MIN).sum()}")
display(df)

BATCH=0  |  30 vectors
  sat != ovf (overflow):  6
  clamped to INT32_MAX:   4
  clamped to INT32_MIN:   2


,A,B,C,D,x,expected_sat,expected_ovf
0,0,1,0,0,-3,9,9
1,2,2,2,2,-2,-10,-10
2,1,2,3,4,2,26,26
3,2,2,2,2,-1,0,0
4,2,3,4,5,1,14,14
5,5,4,3,2,-2,-28,-28
6,0,0,1,0,-10,-10,-10
7,0,0,1,1,0,1,1
8,0,0,100,0,1,100,100
9,0,0,-3,7,5,-8,-8


## Cross-Validate & Export

In [5]:
errors = 0
for _, row in df.iterrows():
    sat, ovf = poly_golden(int(row.A),int(row.B),int(row.C),int(row.D),int(row.x))
    if sat != row.expected_sat or ovf != row.expected_ovf:
        print(f"MISMATCH row {_}: got sat={sat} ovf={ovf}")
        errors += 1

print(f"Cross-validation: {'PASSED' if errors == 0 else f'{errors} FAILURES'} ({len(df)} vectors)")

OUT = "test_vectors.csv"
df.to_csv(OUT, index=False)
print(f"Saved → {OUT}")

Cross-validation: PASSED (30 vectors)
Saved → test_vectors.csv
